# ETL Notebook (Simple)

Reads a CSV from a raw path, filters out rows where `value` is numeric, and writes a Delta output under a processed path organized by `run_date`.

This version avoids widgets/dbutils and resolves parameters from a control table when available.

In [ ]:
from datetime import date
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()


In [ ]:
# Basic configuration
DEFAULT_ENV = 'dev'
DEFAULT_RAW_BASE_PATH = '/Volumes/workspace/default/raw'
DEFAULT_PROCESSED_BASE_PATH = '/Volumes/workspace/default/processed'
DEFAULT_INPUT_FILENAME = 'input.csv'

def get_param(spark, env, key, default=None):
    try:
        df = spark.read.table('workspace.default.control_parameters')
        row = df.filter((df.env == env) & (df.key == key)).select('value').first()
        if row and row.value is not None:
            return row.value
    except Exception:
        pass
    return default

env = DEFAULT_ENV
run_date = get_param(spark, env, 'run_date', default=date.today().strftime('%Y-%m-%d'))
raw_base_path = get_param(spark, env, 'raw_base_path', default=DEFAULT_RAW_BASE_PATH)
processed_base_path = get_param(spark, env, 'processed_base_path', default=DEFAULT_PROCESSED_BASE_PATH)
input_filename = DEFAULT_INPUT_FILENAME

print(f'env={env}, run_date={run_date}')
print(f'raw_base_path={raw_base_path}, processed_base_path={processed_base_path}, input={input_filename}')


In [ ]:
# ETL
input_path = f"{raw_base_path}/{input_filename}"
output_path = f"{processed_base_path}/{run_date}/Notebook"

df = spark.read.option('header', True).csv(input_path)

numeric_regex = r'^[-+]?\d*\.?\d+(e[-+]?\d+)?$'
df_processed = (
    df.filter(col('value').isNotNull())
      .filter(~col('value').rlike(numeric_regex))
)

print(f'Read {df.count()} rows; writing {df_processed.count()} string-only rows to {output_path}')

df_processed.write.format('delta').mode('overwrite').save(output_path)
